# Transformers の pipeline で日本語の固有表現抽出をする

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/19_ner/ner_transformers_ja.ipynb)

NLP クックブック 19-03 ／ 固有表現抽出

Hugging Face Hub にあるファインチューニング済みモデルで、日本語の固有表現抽出をしたい。

In [1]:
!pip install -q fugashi unidic-lite

In [2]:
from transformers import pipeline

ner = pipeline("token-classification", model="llm-book/bert-base-japanese-v3-ner-wikipedia-dataset", aggregation_strategy="simple")
text = "鈴木一郎は1973年に愛知県で生まれ、オリックスとシアトル・マリナーズで活躍した。"
for e in ner(text):
    print(f"{e['word'].replace(' ', ''):\u3000<12}{e['entity_group']:\u3000<8}{e['score']:.3f}")

鈴木一郎　　　　　　　　人名　　　　　　0.999
愛知県　　　　　　　　　地名　　　　　　0.999
オリックス　　　　　　　その他の組織名　0.995
シアトル・マリナーズ　　その他の組織名　0.998


In [3]:
# aggregation_strategy を指定しないとサブワード単位の生の予測 (BIO タグ) が返る
raw = pipeline("token-classification", model="llm-book/bert-base-japanese-v3-ner-wikipedia-dataset")
print([(e["word"], e["entity"]) for e in raw("鈴木一郎は愛知県で生まれた。")])

[('鈴木', 'B-人名'), ('一郎', 'I-人名'), ('愛知', 'B-地名'), ('県', 'I-地名')]


- `aggregation_strategy="simple"` でサブワードを結合し、B-/I- タグをまとめて 1 つの固有表現にしてくれる。
- 日本語 BERT のトークナイザは文字位置を返さないので `start`/`end` は `None` になる。`word` の空白を除いて使う。
- このモデルのラベルは 8 種類: 人名、法人名、政治的組織名、その他の組織名、地名、施設名、製品名、イベント名（ストックマーク社の Wikipedia NER データセット）。
- 多言語モデルなら `tsmatz/xlm-roberta-ner-japanese` などがあり、こちらは `start`/`end` が得られる。